In [1]:
import pandas as pd
import numpy as np
import joblib
import json


day 1 -week 3

In [2]:
ITE_SCORE_PATH = "../data/ite_score.csv"
SCALER_PATH = "../notebooks/confounder_scaler.pkl"
COLUMN_ROLES_PATH = "../notebooks/column_roles.csv"

In [ ]:
'''2. Load ITE Results
This is Dishant's output from `causal/double_ml.py`.'''
df = pd.read_csv(ITE_SCORE_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (5000, 15)
Columns: ['customer_id', 'age', 'income', 'previous_purchases', 'campaign_response', 'customer_tenure_days', 'avg_basket_size', 'channel_online', 'discount', 'purchase', 'treatment', 'ITE', 'ITE_lower', 'ITE_upper', 'segment']


,customer_id,age,income,previous_purchases,campaign_response,customer_tenure_days,avg_basket_size,channel_online,discount,purchase,treatment,ITE,ITE_lower,ITE_upper,segment
0,1053,0.091768,-1.069781,-0.097263,0.065498,-0.978575,-1.892723,0.0,15,0,1,-0.024774,-0.295887,0.246340,Not Persuadable
1,2991,0.693695,0.457315,1.747164,1.360137,0.730126,0.656583,0.0,0,0,0,0.058753,-0.106122,0.223628,Persuadable
2,1789,0.349737,-1.173624,1.132355,0.466836,-1.323725,-0.947376,1.0,0,1,0,0.095262,-0.063939,0.254464,Persuadable
3,765,0.349737,-0.159632,0.517546,0.104338,1.437469,0.053753,1.0,0,1,0,0.069660,-0.171041,0.310362,Persuadable
4,3248,-1.370054,-1.741703,-1.326881,-1.306819,-0.028350,0.600801,0.0,0,1,0,-0.081245,-0.420833,0.258343,Not Persuadable


In [ ]:
''' 3. Derive Actual Discount Levels From the Data
Don't hardcode discount levels — pull them from what the data actually contains.'''
DISCOUNT_LEVELS = sorted(df["discount"].unique().tolist())
print("Discount levels found:", DISCOUNT_LEVELS)

Discount levels found: [0, 5, 10, 15, 20, 25, 30]


In [ ]:
''' 4. Recover Real Order Value (Un-scale `avg_basket_size`)

`avg_basket_size` in `ite_score.csv` is standardized (mean 0, std 1) — we need real ₹ values to compute discount cost, so we inverse-transform using Dishant's saved scaler.'''
try:
    scaler = joblib.load(SCALER_PATH)
    roles = pd.read_csv(COLUMN_ROLES_PATH)
    print("Loaded scaler. Columns it was fit on:")
    print(roles.head(10))
except FileNotFoundError as e:
    print("Could not load scaler:", e)
    print("-> Check the exact scaled feature order in column_roles.csv")

Loaded scaler. Columns it was fit on:
                 column            role  \
0           customer_id      Identifier   
1                   age  Confounder (X)   
2                income  Confounder (X)   
3    previous_purchases  Confounder (X)   
4     campaign_response  Confounder (X)   
5  customer_tenure_days  Confounder (X)   
6       avg_basket_size  Confounder (X)   
7               channel               W   
8              discount   Treatment (T)   
9              purchase     Outcome (Y)   

                                               notes  
0  Not used as a model input, kept for merging re...  
1                              Customer age in years  
2                             Customer income, float  
3                           Count of prior purchases  
4            Historical campaign response rate/score  
5                       Days since customer acquired  
6                         Average basket size, float  
7  Categorical: in_store / online — needs encodi

In [ ]:
'''5. Inverse Transform

Important:the scaler was fit on multiple confounder columns together (`age`, `income`, `previous_purchases`, `campaign_response`, `customer_tenure_days`, `avg_basket_size` — per `double_ml.py`'s `X_features`). You must inverse-transform the full block of scaled columns at once, then pick out `avg_basket_size` — you can't inverse-transform a single column in isolation with a multi-column StandardScaler.
If `column_roles.csv` shows a different column order than below, reorder `scaled_cols` to match.'''

scaled_cols = [
    "age",
    "income",
    "previous_purchases",
    "campaign_response",
    "customer_tenure_days",
    "avg_basket_size",
]

unscaled = scaler.inverse_transform(df[scaled_cols])
unscaled_df = pd.DataFrame(unscaled, columns=[f"{c}_real" for c in scaled_cols], index=df.index)

df["order_value"] = unscaled_df["avg_basket_size_real"].round(2)

print(df[["customer_id", "avg_basket_size", "order_value"]].head())
print("\nReal order_value range:", df["order_value"].min(), "to", df["order_value"].max())

   customer_id  avg_basket_size  order_value
0         1053        -1.892723        18.61
1         2991         0.656583        44.66
2         1789        -0.947376        28.27
3          765         0.053753        38.50
4         3248         0.600801        44.09

Real order_value range: 5.0 to 76.16


In [ ]:
''' 6. Sanity Check
Order values should be positive and plausible. A failure here usually means the scaler's fitted column order doesn't match `scaled_cols` above.'''
negative_orders = (df["order_value"] <= 0).sum()
print(f"Customers with non-positive order_value: {negative_orders}")

if negative_orders > 0:
    print("Column order mismatch likely — verify against column_roles.csv before continuing.")

Customers with non-positive order_value: 0


In [ ]:
## 7. Build the Optimizer Input Structure
optimizer_input = df[["customer_id", "order_value", "ITE", "segment"]].copy()

optimizer_input["discount_options"] = [DISCOUNT_LEVELS for _ in range(len(optimizer_input))]
optimizer_input.head()

,customer_id,order_value,ITE,segment,discount_options
0,1053,18.61,-0.024774,Not Persuadable,"[0, 5, 10, 15, 20, 25, 30]"
1,2991,44.66,0.058753,Persuadable,"[0, 5, 10, 15, 20, 25, 30]"
2,1789,28.27,0.095262,Persuadable,"[0, 5, 10, 15, 20, 25, 30]"
3,765,38.50,0.069660,Persuadable,"[0, 5, 10, 15, 20, 25, 30]"
4,3248,44.09,-0.081245,Not Persuadable,"[0, 5, 10, 15, 20, 25, 30]"


In [ ]:
''' 8. Compute Discount Cost Per Customer Per Discount Level
cost(discount%) = order_value * (discount% / 100)'''
def compute_discount_costs(order_value, discount_levels):
    return [round(order_value * (d / 100), 2) for d in discount_levels]

optimizer_input["discount_cost"] = optimizer_input["order_value"].apply(
    lambda v: compute_discount_costs(v, DISCOUNT_LEVELS)
)
optimizer_input.head()

,customer_id,order_value,ITE,segment,discount_options,predicted_revenue,discount_cost
0,1053,18.61,-0.024774,Not Persuadable,"[0, 5, 10, 15, 20, 25, 30]","[None, None, None, None, None, None, None]","[0.0, 0.93, 1.86, 2.79, 3.72, 4.65, 5.58]"
1,2991,44.66,0.058753,Persuadable,"[0, 5, 10, 15, 20, 25, 30]","[None, None, None, None, None, None, None]","[0.0, 2.23, 4.47, 6.7, 8.93, 11.16, 13.4]"
2,1789,28.27,0.095262,Persuadable,"[0, 5, 10, 15, 20, 25, 30]","[None, None, None, None, None, None, None]","[0.0, 1.41, 2.83, 4.24, 5.65, 7.07, 8.48]"
3,765,38.50,0.069660,Persuadable,"[0, 5, 10, 15, 20, 25, 30]","[None, None, None, None, None, None, None]","[0.0, 1.93, 3.85, 5.77, 7.7, 9.62, 11.55]"
4,3248,44.09,-0.081245,Not Persuadable,"[0, 5, 10, 15, 20, 25, 30]","[None, None, None, None, None, None, None]","[0.0, 2.2, 4.41, 6.61, 8.82, 11.02, 13.23]"


In [ ]:
## 9. Placeholder for predicted_revenue Day 1 defines structure only
optimizer_input["predicted_revenue"] = [[None] * len(DISCOUNT_LEVELS) for _ in range(len(optimizer_input))]

In [ ]:
## 10. Validate Structure
def validate_optimizer_input(data):
    checks = {
        "has_customer_id": "customer_id" in data.columns,
        "has_discount_options": "discount_options" in data.columns,
        "has_discount_cost": "discount_cost" in data.columns,
        "has_predicted_revenue": "predicted_revenue" in data.columns,
        "no_duplicate_customers": data["customer_id"].is_unique,
        "no_missing_order_value": data["order_value"].notnull().all(),
        "discount_options_len_matches_cost": all(
            len(o) == len(c) for o, c in zip(data["discount_options"], data["discount_cost"])
        ),
    }
    for check, passed in checks.items():
        print(f"{'pass' if passed else 'fail'} {check}")
    return all(checks.values())

validate_optimizer_input(optimizer_input)

pass has_customer_id
pass has_discount_options
pass has_discount_cost
pass has_predicted_revenue
pass no_duplicate_customers
pass no_missing_order_value
pass discount_options_len_matches_cost


True

In [64]:
optimizer_input.to_csv("../data/customer_optimizer_input_day1.csv", index=False)
optimizer_input.to_json("../data/customer_optimizer_input_day1.json", orient="records", indent=2)

print("Saved:")
print("../data/customer_optimizer_inputday1.csv")
print("../data/customer_optimizer_inputday1.json")

Saved:
../data/customer_optimizer_inputday1.csv
../data/customer_optimizer_inputday1.json


day 2 -week 3

In [53]:
'''12. Get Baseline Purchase Probability
We need a baseline purchase probability per customer (the "predicted purchase probability" in Dishant's formulation) to compute baseline expected revenue at 0% discount.
This tries a few likely column names from `ite_score.csv` first.
If none of these match what's actually in the file, update `PROB_COL_CANDIDATES` below to the real column name.
If no such column exists at all, 
we fall back to a configurable flat assumption (`DEFAULT_BASELINE_PROB`) — flag this with Dishant/team before relying on it for anything beyond a structural dry run.'''

PROB_COL_CANDIDATES = [
    "baseline_prob",
    "baseline_purchase_prob",
    "propensity_score",
    "predicted_prob",
    "y0_pred",
    "purchase_prob",
]
DEFAULT_BASELINE_PROB = 0.48717  # fallback only — confirm with Dishant/team before using downstream

found_col = next((c for c in PROB_COL_CANDIDATES if c in df.columns), None)

if found_col:
    print(f"Using baseline purchase probability column: '{found_col}'")
    prob_lookup = df.set_index("customer_id")[found_col]
    optimizer_input["baseline_prob"] = optimizer_input["customer_id"].map(prob_lookup)
else:
    print("No baseline purchase-probability column found in ite_score.csv.")
    print(f"   Checked: {PROB_COL_CANDIDATES}")
    print(f"   Falling back to flat DEFAULT_BASELINE_PROB = {DEFAULT_BASELINE_PROB} for all customers.")
    optimizer_input["baseline_prob"] = DEFAULT_BASELINE_PROB

optimizer_input[["customer_id", "baseline_prob"]].head()

No baseline purchase-probability column found in ite_score.csv.
   Checked: ['baseline_prob', 'baseline_purchase_prob', 'propensity_score', 'predicted_prob', 'y0_pred', 'purchase_prob']
   Falling back to flat DEFAULT_BASELINE_PROB = 0.48717 for all customers.


,customer_id,baseline_prob
0,1053,0.48717
1,2991,0.48717
2,1789,0.48717
3,765,0.48717
4,3248,0.48717


In [54]:
'''13. Compute Baseline Revenue (0% Discount)

`baseline_revenue = baseline_prob * order_value`'''
optimizer_input["baseline_revenue"] = (optimizer_input["baseline_prob"] * optimizer_input["order_value"]).round(2)
optimizer_input[["customer_id", "order_value", "baseline_prob", "baseline_revenue"]].head()

,customer_id,order_value,baseline_prob,baseline_revenue
0,1053,18.61,0.48717,9.07
1,2991,44.66,0.48717,21.76
2,1789,28.27,0.48717,13.77
3,765,38.50,0.48717,18.76
4,3248,44.09,0.48717,21.48


In [55]:
'''14. Predicted Revenue Per Discount Level

For each discount level `d`, revenue is baseline revenue plus a share of the customer's `ITE` (treatment effect),
scaled by how far `d` is into the discount range. We use a square-root scaling so the added lift shows diminishing returns as discount increases —
matching the shape of the example table in the schedule doc — rather than a straight linear ramp.

`predicted_revenue(d) = baseline_revenue + ITE * sqrt(d / max_discount)`  (0 at `d = 0`, full `ITE` added at the max discount level)'''
def compute_predicted_revenue(baseline_revenue, ite, discount_levels):
    max_discount = max(discount_levels) if max(discount_levels) > 0 else 1
    revenues = []
    for d in discount_levels:
        lift_fraction = np.sqrt(d / max_discount) if max_discount > 0 else 0
        revenue = baseline_revenue + ite * lift_fraction
        revenues.append(round(float(revenue), 2))
    return revenues

optimizer_input["predicted_revenue"] = optimizer_input.apply(
    lambda row: compute_predicted_revenue(row["baseline_revenue"], row["ITE"], row["discount_options"]),
    axis=1,
)

optimizer_input[["customer_id", "discount_options", "predicted_revenue"]].head()


,customer_id,discount_options,predicted_revenue
0,1053,"[0, 5, 10, 15, 20, 25, 30]","[9.07, 9.06, 9.06, 9.05, 9.05, 9.05, 9.05]"
1,2991,"[0, 5, 10, 15, 20, 25, 30]","[21.76, 21.78, 21.79, 21.8, 21.81, 21.81, 21.82]"
2,1789,"[0, 5, 10, 15, 20, 25, 30]","[13.77, 13.81, 13.82, 13.84, 13.85, 13.86, 13.87]"
3,765,"[0, 5, 10, 15, 20, 25, 30]","[18.76, 18.79, 18.8, 18.81, 18.82, 18.82, 18.83]"
4,3248,"[0, 5, 10, 15, 20, 25, 30]","[21.48, 21.45, 21.43, 21.42, 21.41, 21.41, 21.4]"


In [56]:
## 15. Sanity Checks
def sanity_check(row):
    d0_idx = row["discount_options"].index(min(row["discount_options"]))
    revenue_at_min_discount = row["predicted_revenue"][d0_idx]
    matches_baseline = abs(revenue_at_min_discount - row["baseline_revenue"]) < 0.01
    all_non_negative = all(r >= 0 for r in row["predicted_revenue"])
    return pd.Series({"matches_baseline_at_min_discount": matches_baseline, "all_non_negative": all_non_negative})

checks = optimizer_input.apply(sanity_check, axis=1)
print("Rows where predicted_revenue at the lowest discount doesn't match baseline_revenue:",
      (~checks["matches_baseline_at_min_discount"]).sum())
print("Rows with any negative predicted_revenue:", (~checks["all_non_negative"]).sum())

Rows where predicted_revenue at the lowest discount doesn't match baseline_revenue: 0
Rows with any negative predicted_revenue: 0


In [57]:
## 16. Validate Structure 
def validate_day2_structure(data):
    checks = {
        "has_customer_id": "customer_id" in data.columns,
        "has_discount_options": "discount_options" in data.columns,
        "has_discount_cost": "discount_cost" in data.columns,
        "has_predicted_revenue": "predicted_revenue" in data.columns,
        "no_duplicate_customers": data["customer_id"].is_unique,
        "no_missing_predicted_revenue": data["predicted_revenue"].apply(lambda r: all(v is not None for v in r)).all(),
        "predicted_revenue_len_matches_options": all(
            len(o) == len(r) for o, r in zip(data["discount_options"], data["predicted_revenue"])
        ),
    }
    for check, passed in checks.items():
        print(f"{'pass' if passed else 'fail'} {check}")
    return all(checks.values())

validate_day2_structure(optimizer_input)

pass has_customer_id
pass has_discount_options
pass has_discount_cost
pass has_predicted_revenue
pass no_duplicate_customers
pass no_missing_predicted_revenue
pass predicted_revenue_len_matches_options


True

In [58]:
## 17. Save Final Output for Day 3 / Handoff to Dishant
final_cols = ["customer_id", "order_value", "segment", "ITE", "discount_options", "discount_cost", "predicted_revenue"]
optimizer_input_day2 = optimizer_input[final_cols].copy()

optimizer_input_day2.to_csv("../data/customer_optimizer_input_day2.csv", index=False)
optimizer_input_day2.to_json("../data/customer_optimizer_input_day2.json", orient="records", indent=2)

print("Saved:")
print(f" - {"../data/customer_optimizer_input_day2.csv"}")
print(f" - {"../data/customer_optimizer_input_day2.json"}")

Saved:
 - ../data/customer_optimizer_input_day2.csv
 - ../data/customer_optimizer_input_day2.json


day 3 - week 3

In [59]:
'''This turns the per-customer `predicted_revenue` / `discount_cost` list-columns into two aligned 2D numpy arrays that Dishant's SciPy optimizer (Day 4) can index directly instead of looping over dataframe rows:

    revenue_matrix[i, j] = predicted revenue for customer i at discount level j
    cost_matrix[i, j]    = discount cost for customer i at discount level j

Row order = `customer_ids` order; column order = `DISCOUNT_LEVELS` order — both matrices share both.'''

OUT_NPZ_PATH = "../data/customer_optimizer_matrices_final.npz"
OUT_JSON_PATH = "../data/customer_optimizer_matrices_final.json"
OUT_CUSTOMER_INDEX_CSV_PATH = "../data/customer_optimizer_index_final.csv"

In [60]:
'''## 19. Build the Matrices

`optimizer_input["discount_options"]` was built in Day 1 as the same shared `DISCOUNT_LEVELS` list for every customer (see cell 7),
so every customer's `predicted_revenue` and `discount_cost` lists are already guaranteed to be the same length and in the same order — no re-validation needed. Stack them directly.'''
def build_matrices(data, discount_levels):
    """Stack each customer's per-discount list into a (n_customers x n_discounts) numpy array."""
    ids = data["customer_id"].to_numpy()
    revenue = np.array(data["predicted_revenue"].tolist(), dtype=float)
    cost = np.array(data["discount_cost"].tolist(), dtype=float)

    assert revenue.shape == cost.shape == (len(data), len(discount_levels))
    return ids, revenue, cost

customer_ids, revenue_matrix, cost_matrix = build_matrices(optimizer_input, DISCOUNT_LEVELS)

print(f"revenue_matrix shape: {revenue_matrix.shape}  (customers x discount levels)")
print(f"cost_matrix shape:    {cost_matrix.shape}  (customers x discount levels)")

revenue_matrix shape: (5000, 7)  (customers x discount levels)
cost_matrix shape:    (5000, 7)  (customers x discount levels)


In [61]:
'''## 20. Sanity Checks 
These check the matrices themselves (NaNs, sign, shape)'''
def sanity_check_matrices(ids, revenue, cost, discount_levels):
    checks = {
        "same_row_count": len(ids) == revenue.shape[0] == cost.shape[0],
        "same_col_count_as_discount_levels": revenue.shape[1] == len(discount_levels) == cost.shape[1],
        "no_nan_in_revenue": not np.isnan(revenue).any(),
        "no_nan_in_cost": not np.isnan(cost).any(),
        "revenue_non_negative": (revenue >= 0).all(),
        "cost_non_negative": (cost >= 0).all(),
        "cost_zero_at_zero_discount": (
            cost[:, discount_levels.index(0)] == 0
        ).all() if 0 in discount_levels else True,
        "unique_customer_ids": len(set(ids)) == len(ids),
    }
    for check, passed in checks.items():
        print(f"{'pass' if passed else 'fail'} {check}")
    return all(checks.values())

sanity_check_matrices(customer_ids, revenue_matrix, cost_matrix, DISCOUNT_LEVELS)

pass same_row_count
pass same_col_count_as_discount_levels
pass no_nan_in_revenue
pass no_nan_in_cost
pass revenue_non_negative
pass cost_non_negative
pass cost_zero_at_zero_discount
pass unique_customer_ids


True

In [62]:
'''21. Preview
In the same format as the schedule doc's example: `C1 -> [100, 108, 115, 118]; C2 -> [80, 95, 110, 107]`'''
preview = "; ".join(
    f"{cid} -> [{', '.join(str(round(v)) for v in row)}]"
    for cid, row in zip(customer_ids[:3], revenue_matrix[:3])
)
print(preview)

1053 -> [9, 9, 9, 9, 9, 9, 9]; 2991 -> [22, 22, 22, 22, 22, 22, 22]; 1789 -> [14, 14, 14, 14, 14, 14, 14]


In [63]:
## 22. Save Matrices for dishant
np.savez(
    OUT_NPZ_PATH,
    customer_ids=customer_ids,
    discount_levels=np.array(DISCOUNT_LEVELS, dtype=float),
    revenue_matrix=revenue_matrix,
    cost_matrix=cost_matrix,
)

# .json — human-readable mirror, for review / debugging
payload = {
    "discount_levels": list(DISCOUNT_LEVELS),
    "customers": [
        {
            "customer_id": str(cid),
            "predicted_revenue": row_rev.tolist(),
            "discount_cost": row_cost.tolist(),
        }
        for cid, row_rev, row_cost in zip(customer_ids, revenue_matrix, cost_matrix)
    ],
}
with open(OUT_JSON_PATH, "w") as f:
    json.dump(payload, f, indent=2)

# customer index csv — row i of the matrices <-> this customer_id, so Dishant's
# index-based optimizer output can be mapped back to customer_id
pd.DataFrame({"row_index": range(len(customer_ids)), "customer_id": customer_ids}).to_csv(
    OUT_CUSTOMER_INDEX_CSV_PATH, index=False
)

print("Saved:")
print(f" - {OUT_NPZ_PATH}  (revenue_matrix, cost_matrix, customer_ids, discount_levels)")
print(f" - {OUT_JSON_PATH}")
print(f" - {OUT_CUSTOMER_INDEX_CSV_PATH}")

Saved:
 - ../data/customer_optimizer_matrices_final.npz  (revenue_matrix, cost_matrix, customer_ids, discount_levels)
 - ../data/customer_optimizer_matrices_final.json
 - ../data/customer_optimizer_index_final.csv


day4- week 3

In [65]:
'''23. Load Day 3 Output From Disk
Reloading from the saved `.npz` (not from the in-memory `revenue_matrix`/`cost_matrix` variables above) on purpose — this is what actually gets tested, since Dishant will load the file fresh in his own script, not share this notebook's session.'''
def load_day3_matrices(npz_path=OUT_NPZ_PATH):
    data = np.load(npz_path, allow_pickle=True)
    return (
        data["customer_ids"],
        data["discount_levels"],
        data["revenue_matrix"],
        data["cost_matrix"],
    )

loaded_customer_ids, loaded_discount_levels, loaded_revenue_matrix, loaded_cost_matrix = load_day3_matrices()
print("Reloaded from disk:", loaded_revenue_matrix.shape, "customers x discount levels")

Reloaded from disk: (5000, 7) customers x discount levels


In [67]:
''' Validate Before Handoff
Final pre-handoff checks. If any of these fail, the problem is upstream (Day 1-3) and must be fixed there — Day 4 should never hand Dishant data it hasn't confirmed is clean.'''
def validate_for_handoff(customer_ids, discount_levels, revenue_matrix, cost_matrix):
    n_customers = len(customer_ids)
    n_levels = len(discount_levels)

    checks = {
        "revenue_matrix shape matches (customers, levels)": revenue_matrix.shape == (n_customers, n_levels),
        "cost_matrix shape matches (customers, levels)": cost_matrix.shape == (n_customers, n_levels),
        "no NaNs in revenue_matrix": not np.isnan(revenue_matrix).any(),
        "no NaNs in cost_matrix": not np.isnan(cost_matrix).any(),
        "revenue_matrix non-negative": (revenue_matrix >= 0).all(),
        "cost_matrix non-negative": (cost_matrix >= 0).all(),
        "discount_levels sorted ascending": list(discount_levels) == sorted(discount_levels),
        "discount_levels starts at 0": discount_levels[0] == 0,
        "unique customer_ids": len(set(customer_ids)) == n_customers,
        "at least one customer present": n_customers > 0,
    }

    print("Day 4 handoff validation:")
    for check, passed in checks.items():
        print(f"{'pass'if passed else 'fail'} {check}")

    if not all(checks.values()):
        failed = [c for c, ok in checks.items() if not ok]
        raise AssertionError(f"Failed Day 4 handoff checks: {failed}")

    print(f"\nReady for handoff: {n_customers} customers x {n_levels} discount levels.")
    return True

validate_for_handoff(loaded_customer_ids, loaded_discount_levels, loaded_revenue_matrix, loaded_cost_matrix)

Day 4 handoff validation:
pass revenue_matrix shape matches (customers, levels)
pass cost_matrix shape matches (customers, levels)
pass no NaNs in revenue_matrix
pass no NaNs in cost_matrix
pass revenue_matrix non-negative
pass cost_matrix non-negative
pass discount_levels sorted ascending
pass discount_levels starts at 0
pass unique customer_ids
pass at least one customer present

Ready for handoff: 5000 customers x 7 discount levels.


True

In [69]:
'''25. The Single Import Point for Dishant
This is the function Dishant's `prescriptive_optimization.py` should import and call. It returns exactly what his SciPy optimizer needs — he does not need to touch any Day 1-3 code.'''
def get_optimizer_inputs(npz_path=OUT_NPZ_PATH):
    """
    Usage in Dishant's file:
        from princy_day4_connect_to_optimizer import get_optimizer_inputs
        customer_ids, discount_levels, revenue_matrix, cost_matrix = get_optimizer_inputs()

    Returns
    -------
    customer_ids : np.ndarray, shape (n_customers,)
    discount_levels : np.ndarray, shape (n_discount_levels,)  -- PERCENT, e.g. [0, 5, 10, ..., 30]
    revenue_matrix : np.ndarray, shape (n_customers, n_discount_levels)
    cost_matrix : np.ndarray, shape (n_customers, n_discount_levels)
    """
    ids, levels, revenue, cost = load_day3_matrices(npz_path)
    validate_for_handoff(ids, levels, revenue, cost)
    return ids, levels, revenue, cost

# End-to-end proof this works exactly as Dishant will call it
handoff_customer_ids, handoff_discount_levels, handoff_revenue_matrix, handoff_cost_matrix = get_optimizer_inputs()

print("\ndiscount_levels (percent):", handoff_discount_levels.tolist())
print("\nSample handoff preview (first 3 customers):")
for cid, rev_row, cost_row in zip(handoff_customer_ids[:3], handoff_revenue_matrix[:3], handoff_cost_matrix[:3]):
    print(f"  {cid}: revenue={rev_row.tolist()}  cost={cost_row.tolist()}")

Day 4 handoff validation:
pass revenue_matrix shape matches (customers, levels)
pass cost_matrix shape matches (customers, levels)
pass no NaNs in revenue_matrix
pass no NaNs in cost_matrix
pass revenue_matrix non-negative
pass cost_matrix non-negative
pass discount_levels sorted ascending
pass discount_levels starts at 0
pass unique customer_ids
pass at least one customer present

Ready for handoff: 5000 customers x 7 discount levels.

discount_levels (percent): [0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0]

Sample handoff preview (first 3 customers):
  1053: revenue=[9.07, 9.06, 9.06, 9.05, 9.05, 9.05, 9.05]  cost=[0.0, 0.93, 1.86, 2.79, 3.72, 4.65, 5.58]
  2991: revenue=[21.76, 21.78, 21.79, 21.8, 21.81, 21.81, 21.82]  cost=[0.0, 2.23, 4.47, 6.7, 8.93, 11.16, 13.4]
  1789: revenue=[13.77, 13.81, 13.82, 13.84, 13.85, 13.86, 13.87]  cost=[0.0, 1.41, 2.83, 4.24, 5.65, 7.07, 8.48]
